# Student Placement Prediction System
Predicts whether a student will be placed using academic and skill-based features, comparing Logistic Regression, Decision Tree, and Random Forest.

## Step 1: Import Libraries

In [ ]:
import pickle

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


## Step 2: Load Dataset

In [ ]:
DATA_PATH = "dataset.csv"
TARGET_COLUMN = "PlacementStatus"

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


## Step 3: Check Missing Values

In [ ]:
df.isnull().sum()


## Step 4: Handle Missing Values
Numeric columns are filled with the column mean; categorical columns are filled with the most frequent value (mode).

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isnull().sum()


## Step 5: Encode Categorical Columns
Machine learning models need numeric input, so every categorical column (including the target) is label-encoded.

In [ ]:
encoders = {}
for col in categorical_cols:
    if col == TARGET_COLUMN:
        continue
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

if df[TARGET_COLUMN].dtype == "object":
    target_encoder = LabelEncoder()
    df[TARGET_COLUMN] = target_encoder.fit_transform(df[TARGET_COLUMN])
else:
    target_encoder = None

df.head()


## Step 6: Split Features and Target

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
feature_names = X.columns.tolist()
feature_names


## Step 7: Feature Scaling

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Step 8: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape


## Step 9: Train and Compare Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f"\n{name} Accuracy: {acc * 100:.2f}%")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))


## Step 10: Select Best Model

In [ ]:
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
print(f"Best model: {best_model_name} ({results[best_model_name] * 100:.2f}% accuracy)")


## Step 11: Feature Importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": best_model.feature_importances_,
    }).sort_values(by="Importance", ascending=False)
    display(importance_df)


## Step 12: Placement Probability per Student

In [ ]:
if hasattr(best_model, "predict_proba"):
    probabilities = best_model.predict_proba(X_test)[:, 1]
    prob_df = pd.DataFrame({
        "Predicted_Placement": best_model.predict(X_test),
        "Placement_Probability": probabilities,
    })
    prob_df.head()


## Step 13: Save the Best Model

In [ ]:
with open("model.pkl", "wb") as f:
    pickle.dump({
        "model": best_model,
        "scaler": scaler,
        "encoders": encoders,
        "target_encoder": target_encoder,
        "feature_names": feature_names,
    }, f)

print("Model saved to model.pkl")
